In [1]:
import sys
import torch
import platform

print("ENVIRONMENT INFORMATION")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for the inference-time benchmark."
    )

device = torch.device("cuda:0")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("Operating system:", platform.platform())



ENVIRONMENT INFORMATION
Python: 3.10.12
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
CUDA version: 12.8
Operating system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35


In [2]:
# 2. Benchmark Configuration & Reproducibility

import os
import gc
import time
import random

import numpy as np
import pandas as pd
import torch


# Benchmark configuration

SEED = 123

INPUT_HEIGHT = 352
INPUT_WIDTH = 352

BATCH_SIZE = 1
INPUT_CHANNELS = 1

WARMUP_RUNS = 20
MEASURED_RUNS = 100
TIMING_TRIALS = 3

# Use the device defined in Cell 1
DEVICE = device

OUTPUT_DIR = "./complexity_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Reproducibility

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# FP32 configuration

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

# Display configuration

print("BENCHMARK CONFIGURATION")

print(
    "Input:",
    (
        BATCH_SIZE,
        INPUT_CHANNELS,
        INPUT_HEIGHT,
        INPUT_WIDTH
    )
)

print("Device:", DEVICE)
print("Precision: FP32")
print("Seed:", SEED)
print("Warmup runs:", WARMUP_RUNS)
print("Measured runs:", MEASURED_RUNS)
print("Timing trials:", TIMING_TRIALS)
print("Output directory:", OUTPUT_DIR)



BENCHMARK CONFIGURATION
Input: (1, 1, 352, 352)
Device: cuda:0
Precision: FP32
Seed: 123
Warmup runs: 20
Measured runs: 100
Timing trials: 3
Output directory: ./complexity_results


In [3]:
def count_parameters(model):
    """
    Count model parameters.

    Returns
    -------
    total_params : int
        All registered parameters.

    trainable_params : int
        Parameters with requires_grad=True.
    """

    total_params = sum(
        p.numel() for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total_params, trainable_params

In [3]:
# ============================================================
# 4. GPU Inference Time Measurement
# ============================================================

def measure_inference_time(
    model,
    input_tensor,
    warmup_runs=WARMUP_RUNS,
    measured_runs=MEASURED_RUNS,
    timing_trials=TIMING_TRIALS
):
    """
    Measure GPU forward-pass latency using CUDA Events.

    Timing protocol:
        - Batch size: 1
        - FP32
        - Warmup: 20 runs
        - Measurement: 100 runs per trial
        - Trials: 3

    Returns:
        Mean latency across trials (ms)
        Standard deviation of trial means (ms)
        Median latency across all runs (ms)
        Total measured forward passes
    """

    if not input_tensor.is_cuda:
        raise ValueError(
            "Input tensor must be on CUDA."
        )

    if next(model.parameters()).device != input_tensor.device:
        raise ValueError(
            "Model and input tensor must be on the same device."
        )

    model.eval()

    trial_means = []
    all_times = []

    # Create CUDA Events once
    start_event = torch.cuda.Event(
        enable_timing=True
    )

    end_event = torch.cuda.Event(
        enable_timing=True
    )

    with torch.inference_mode():

        # ----------------------------------------------------
        # 1. Warmup
        # ----------------------------------------------------

        for _ in range(warmup_runs):
            _ = model(input_tensor)

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # 2. Timing trials
        # ----------------------------------------------------

        for trial in range(timing_trials):

            trial_times = []

            for _ in range(measured_runs):

                start_event.record()

                _ = model(input_tensor)

                end_event.record()

                end_event.synchronize()

                elapsed_ms = start_event.elapsed_time(
                    end_event
                )

                trial_times.append(elapsed_ms)

            trial_mean = float(
                np.mean(trial_times)
            )

            trial_means.append(trial_mean)
            all_times.extend(trial_times)

            print(
                f"Trial {trial + 1}: "
                f"{trial_mean:.4f} ms"
            )

    return {
        "inference_mean_ms": float(
            np.mean(trial_means)
        ),
        "inference_std_ms": float(
            np.std(trial_means, ddof=1)
        ),
        "inference_median_ms": float(
            np.median(all_times)
        ),
        "measured_passes": len(all_times),
    }

In [4]:
# ============================================================
# 7. Model Code Directories — Safe Configuration
# ============================================================

import sys
import importlib.util
import inspect

from pathlib import Path

# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/sudad/Deburring/Deblurring-SOTA"
)

MODEL_CODE_DIRS = {
    "FFTformer": PROJECT_ROOT / "FFTformer",
    "MaIR": PROJECT_ROOT / "MaIR",
    "NAFNet": PROJECT_ROOT / "NAFNet",
    "SwinIR-M": PROJECT_ROOT / "SwinIR",
    "Proposed": PROJECT_ROOT / "Proposed",
}

# ------------------------------------------------------------
# 2. Verify directories
# ------------------------------------------------------------

for name, directory in MODEL_CODE_DIRS.items():

    if not directory.is_dir():
        raise FileNotFoundError(
            f"{name} directory not found: {directory}"
        )

    print(f"[OK] {name}: {directory}")

# ------------------------------------------------------------
# 3. Configure paths for other models
# ------------------------------------------------------------

for name in [
    "FFTformer",
    "MaIR",
    "NAFNet",
    "SwinIR-M"
]:

    directory = MODEL_CODE_DIRS[name]

    if str(directory) not in sys.path:
        sys.path.append(str(directory))

# ------------------------------------------------------------
# 4. Import Proposed from its exact file
# ------------------------------------------------------------

PROPOSED_MODULE_PATH = (
    MODEL_CODE_DIRS["Proposed"] / "Module.py"
)

if not PROPOSED_MODULE_PATH.is_file():
    raise FileNotFoundError(
        f"Proposed Module.py not found: "
        f"{PROPOSED_MODULE_PATH}"
    )

spec = importlib.util.spec_from_file_location(
    "proposed_architecture_module",
    str(PROPOSED_MODULE_PATH)
)

if spec is None or spec.loader is None:
    raise ImportError(
        "Could not create Proposed module specification."
    )

proposed_module = importlib.util.module_from_spec(
    spec
)

sys.modules[spec.name] = proposed_module

spec.loader.exec_module(proposed_module)

MedMultiwaveletDeblurringGenerator = (
    proposed_module.MedMultiwaveletDeblurringGenerator
)

# ------------------------------------------------------------
# 5. Verify imported architecture
# ------------------------------------------------------------

actual_path = Path(
    inspect.getfile(
        MedMultiwaveletDeblurringGenerator
    )
).resolve()

expected_path = PROPOSED_MODULE_PATH.resolve()

if actual_path != expected_path:
    raise RuntimeError(
        "Incorrect Proposed architecture imported!\n"
        f"Expected: {expected_path}\n"
        f"Actual: {actual_path}"
    )

print("\n" + "=" * 65)
print("PROPOSED ARCHITECTURE VERIFIED")
print("=" * 65)

print("Source:", actual_path)

print(
    "Constructor:",
    inspect.signature(
        MedMultiwaveletDeblurringGenerator
    )
)

print("\n[OK] Model code directories configured safely.")

[OK] FFTformer: /home/sudad/Deburring/Deblurring-SOTA/FFTformer
[OK] MaIR: /home/sudad/Deburring/Deblurring-SOTA/MaIR
[OK] NAFNet: /home/sudad/Deburring/Deblurring-SOTA/NAFNet
[OK] SwinIR-M: /home/sudad/Deburring/Deblurring-SOTA/SwinIR
[OK] Proposed: /home/sudad/Deburring/Deblurring-SOTA/Proposed


/home/sudad/mamba_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



PROPOSED ARCHITECTURE VERIFIED
Source: /home/sudad/Deburring/Deblurring-SOTA/Proposed/Module.py
Constructor: (in_ch: 'int' = 1, out_ch: 'int' = 1, wavelet_type: 'str' = 'ghm', band_feature_channels: 'int' = 40, feature_channels: 'int' = 192, cnn_blocks: 'int' = 8, mamba_blocks: 'int' = 8, branch_mode: 'str' = 'dual', fusion_type: 'str' = 'gated', use_alignment: 'bool' = True, use_band_refinement: 'bool' = True, use_full_res_refinement: 'bool' = True)

[OK] Model code directories configured safely.


In [5]:
# ============================================================
# 8. Import Model Architectures
# ============================================================

import inspect
from pathlib import Path

# ------------------------------------------------------------
# 1. Import SOTA architectures
# ------------------------------------------------------------

from FFTformer_Module import build_fftformer
from MaIR_Module import build_mair
from NAFNet_Module import build_nafnet
from SwinIR_Module import build_swinir

# ------------------------------------------------------------
# 2. Verify imported module locations
# ------------------------------------------------------------

MODEL_BUILDERS = {
    "FFTformer": build_fftformer,
    "MaIR": build_mair,
    "NAFNet": build_nafnet,
    "SwinIR-M": build_swinir,
    "Proposed": MedMultiwaveletDeblurringGenerator,
}

for name, builder in MODEL_BUILDERS.items():

    actual_path = Path(
        inspect.getfile(builder)
    ).resolve()

    expected_dir = MODEL_CODE_DIRS[name].resolve()

    if actual_path.parent != expected_dir:
        raise RuntimeError(
            f"Incorrect architecture imported for {name}!\n"
            f"Expected directory: {expected_dir}\n"
            f"Actual file: {actual_path}"
        )

    print(f"[OK] {name}: {actual_path}")

print("\nAll five model architectures verified.")

Official MaIRUNet loaded successfully.
[OK] FFTformer: /home/sudad/Deburring/Deblurring-SOTA/FFTformer/FFTformer_Module.py
[OK] MaIR: /home/sudad/Deburring/Deblurring-SOTA/MaIR/MaIR_Module.py
[OK] NAFNet: /home/sudad/Deburring/Deblurring-SOTA/NAFNet/NAFNet_Module.py
[OK] SwinIR-M: /home/sudad/Deburring/Deblurring-SOTA/SwinIR/SwinIR_Module.py
[OK] Proposed: /home/sudad/Deburring/Deblurring-SOTA/Proposed/Module.py

All five model architectures verified.


In [6]:
# 9---============================================================
# Checkpoint Paths — Five Models
# ============================================================

from pathlib import Path

SOTA_RESULTS = Path(
    "/home/sudad/Deburring/Deblurring-SOTA/Results"
)

CHECKPOINT_PATHS = {

    "FFTformer": (
        SOTA_RESULTS
        / "123-FFTformer_MotionDeblurring_unified"
        / "best_ema_generator.pt"
    ),

    "MaIR": (
        SOTA_RESULTS
        / "123-MaIR_MotionDeblurring_unified"
        / "best_ema_generator.pt"
    ),

    "NAFNet": (
        SOTA_RESULTS
        / "123-NAFNet_MotionDeblurring_unified"
        / "best_ema_generator.pt"
    ),

    "SwinIR-M": (
        SOTA_RESULTS
        / "123-SwinIR_M_MotionDeblurring_unified"
        / "best_ema_generator.pt"
    ),

    "Proposed": (
        SOTA_RESULTS
        /"123-Proposed"
        /"best_ema_generator.pt"
    ),
}

print("=" * 65)
print("CHECKPOINT VERIFICATION")
print("=" * 65)

for model_name, checkpoint_path in CHECKPOINT_PATHS.items():

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"{model_name}: {checkpoint_path}"
        )

    print(f"[OK] {model_name}")

print("\nAll five checkpoints found successfully.")

CHECKPOINT VERIFICATION
[OK] FFTformer
[OK] MaIR
[OK] NAFNet
[OK] SwinIR-M
[OK] Proposed

All five checkpoints found successfully.


In [7]:
# ============================================================
# 10. Create Model Architecture
# ============================================================

def create_model(model_name):

    if model_name == "FFTformer":
        return build_fftformer()

    elif model_name == "MaIR":
        return build_mair(
            patch_size=88,
            batch_size=1
        )

    elif model_name == "NAFNet":
        return build_nafnet()

    elif model_name == "SwinIR-M":
        return build_swinir()

    elif model_name == "Proposed":
        return MedMultiwaveletDeblurringGenerator()

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

In [8]:
# ============================================================
# 14. Load Trained Model — Five Models
# ============================================================

import torch
from pathlib import Path


def load_trained_model(model_name):

    print(f"\nLoading {model_name}...")

    # --------------------------------------------------------
    # 1. Verify model name
    # --------------------------------------------------------

    if model_name not in CHECKPOINT_PATHS:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    # --------------------------------------------------------
    # 2. Create original model architecture
    # --------------------------------------------------------

    model = create_model(model_name)

    # --------------------------------------------------------
    # 3. Locate checkpoint
    # --------------------------------------------------------

    checkpoint_path = Path(
        CHECKPOINT_PATHS[model_name]
    )

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    # --------------------------------------------------------
    # 4. Load checkpoint on CPU
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    if not isinstance(checkpoint, dict):
        raise TypeError(
            f"{model_name}: Unexpected checkpoint format."
        )

    # --------------------------------------------------------
    # 5. Select correct generator weights
    # --------------------------------------------------------

    if model_name == "Proposed":
        weights_key = "gen"
    else:
        weights_key = "generator"

    if weights_key not in checkpoint:
        raise KeyError(
            f"{model_name}: Missing weights key "
            f"'{weights_key}'.\n"
            f"Available keys: {list(checkpoint.keys())}"
        )

    state_dict = checkpoint[weights_key]

    # --------------------------------------------------------
    # 6. Verify architecture compatibility
    # --------------------------------------------------------

    model.load_state_dict(
        state_dict,
        strict=True
    )

    # --------------------------------------------------------
    # 7. Move model to GPU
    # --------------------------------------------------------

    model = model.to(
        device=DEVICE,
        dtype=torch.float32
    )

    model.eval()

    # --------------------------------------------------------
    # 8. Report loading information
    # --------------------------------------------------------

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print(f"[OK] {model_name}: checkpoint loaded successfully.")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Weights key: {weights_key}")
    print("Strict loading: SUCCESS")
    print(f"Parameters: {total_params / 1e6:.6f} M")

    if "epoch" in checkpoint:
        print(f"Epoch: {checkpoint['epoch']}")

    if "weights_type" in checkpoint:
        print(f"Weights type: {checkpoint['weights_type']}")

    # --------------------------------------------------------
    # 9. Release checkpoint memory
    # --------------------------------------------------------

    del checkpoint
    del state_dict

    return model

In [13]:
MODEL_NAMES = ["FFTformer", "MaIR", "NAFNet", "SwinIR-M", "Proposed"]
PARAMETERS_CSV = os.path.join(OUTPUT_DIR, "model_parameters.csv")

parameter_rows = []
for name in MODEL_NAMES:
    model = None
    try:
        model = load_trained_model(name)
        total, trainable = count_parameters(model)
        parameter_rows.append({
            "model": name,
            "parameters": total,
            "parameters_m": total / 1e6,
            "trainable_parameters": trainable,
            "trainable_parameters_m": trainable / 1e6,
        })
    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

parameters_df = pd.DataFrame(parameter_rows)
parameters_df.to_csv(PARAMETERS_CSV, index=False)
print(parameters_df.to_string(index=False))
print("Saved:", os.path.abspath(PARAMETERS_CSV))


Loading FFTformer...
[OK] FFTformer: checkpoint loaded successfully.
Checkpoint: /home/sudad/Deburring/Deblurring-SOTA/Results/123-FFTformer_MotionDeblurring_unified/best_ema_generator.pt
Weights key: generator
Strict loading: SUCCESS
Parameters: 16.558746 M
Epoch: 30
Weights type: ema


NameError: name 'count_parameters' is not defined

In [9]:
from pathlib import Path

TIMING_DIR = Path(OUTPUT_DIR) / "inference_time"
TIMING_DIR.mkdir(parents=True, exist_ok=True)


def benchmark_inference(model_name):
    if model_name not in CHECKPOINT_PATHS:
        raise ValueError(f"Unknown model: {model_name}")

    model = None
    x = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        model = load_trained_model(model_name)
        model.eval()
        x = torch.randn(
            BATCH_SIZE, INPUT_CHANNELS, INPUT_HEIGHT, INPUT_WIDTH,
            device=DEVICE, dtype=torch.float32
        )

        with torch.inference_mode():
            y = model(x)
            if not isinstance(y, torch.Tensor) or y.shape != x.shape:
                raise RuntimeError(f"Invalid output shape for {model_name}")
            if not torch.isfinite(y).all().item():
                raise RuntimeError(f"Non-finite output for {model_name}")
            del y
        torch.cuda.synchronize()

        metrics = measure_inference_time(
            model, x,
            warmup_runs=WARMUP_RUNS,
            measured_runs=MEASURED_RUNS,
            timing_trials=TIMING_TRIALS,
        )
        row = {
            "model": model_name,
            "inference_mean_ms": metrics["inference_mean_ms"],
            "inference_std_ms": metrics["inference_std_ms"],
            "inference_median_ms": metrics["inference_median_ms"],
            "measured_passes": metrics["measured_passes"],
            "input_shape": f"{BATCH_SIZE}x{INPUT_CHANNELS}x{INPUT_HEIGHT}x{INPUT_WIDTH}",
            "precision": "FP32",
            "gpu": torch.cuda.get_device_name(0),
            "pytorch": torch.__version__,
            "cuda": torch.version.cuda,
            "warmup_runs": WARMUP_RUNS,
            "measured_runs": MEASURED_RUNS,
            "timing_trials": TIMING_TRIALS,
            "seed": SEED,
        }

        single_path = TIMING_DIR / f"{model_name.replace('-', '_')}_inference.csv"
        pd.DataFrame([row]).to_csv(single_path, index=False)

        summary_path = TIMING_DIR / "inference_time_summary.csv"
        if summary_path.exists():
            summary = pd.read_csv(summary_path)
            summary = summary[summary["model"] != model_name]
            summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
        else:
            summary = pd.DataFrame([row])
        order = {name: i for i, name in enumerate(CHECKPOINT_PATHS)}
        summary = summary.sort_values("model", key=lambda col: col.map(order))
        temp_path = summary_path.with_suffix(".tmp")
        summary.to_csv(temp_path, index=False)
        os.replace(temp_path, summary_path)

        print(f"{model_name}: {row['inference_mean_ms']:.4f} ± {row['inference_std_ms']:.4f} ms")
        print("Saved:", single_path)
        print("Summary:", summary_path)
        return row
    finally:
        if x is not None:
            del x
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

In [13]:
benchmark_inference("FFTformer")


Loading FFTformer...
[OK] FFTformer: checkpoint loaded successfully.
Checkpoint: /home/sudad/Deburring/Deblurring-SOTA/Results/123-FFTformer_MotionDeblurring_unified/best_ema_generator.pt
Weights key: generator
Strict loading: SUCCESS
Parameters: 16.558746 M
Epoch: 30
Weights type: ema
Trial 1: 433.5386 ms
Trial 2: 438.9435 ms
Trial 3: 441.7628 ms
FFTformer: 438.0816 ± 4.1793 ms
Saved: complexity_results/inference_time/FFTformer_inference.csv
Summary: complexity_results/inference_time/inference_time_summary.csv


{'model': 'FFTformer',
 'inference_mean_ms': 438.08162404378254,
 'inference_std_ms': 4.179302147774101,
 'inference_median_ms': 432.81105041503906,
 'measured_passes': 300,
 'input_shape': '1x1x352x352',
 'precision': 'FP32',
 'gpu': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU',
 'pytorch': '2.11.0+cu128',
 'cuda': '12.8',
 'warmup_runs': 20,
 'measured_runs': 100,
 'timing_trials': 3,
 'seed': 123}

In [12]:
benchmark_inference("MaIR")


Loading MaIR...
[OK] MaIR: checkpoint loaded successfully.
Checkpoint: /home/sudad/Deburring/Deblurring-SOTA/Results/123-MaIR_MotionDeblurring_unified/best_ema_generator.pt
Weights key: generator
Strict loading: SUCCESS
Parameters: 26.290704 M
Epoch: 30
Weights type: ema
Trial 1: 971.7193 ms
Trial 2: 973.3864 ms
Trial 3: 966.2383 ms
MaIR: 970.4480 ± 3.7398 ms
Saved: complexity_results/inference_time/MaIR_inference.csv
Summary: complexity_results/inference_time/inference_time_summary.csv


{'model': 'MaIR',
 'inference_mean_ms': 970.4479876708983,
 'inference_std_ms': 3.7398063938600727,
 'inference_median_ms': 968.5534973144531,
 'measured_passes': 300,
 'input_shape': '1x1x352x352',
 'precision': 'FP32',
 'gpu': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU',
 'pytorch': '2.11.0+cu128',
 'cuda': '12.8',
 'warmup_runs': 20,
 'measured_runs': 100,
 'timing_trials': 3,
 'seed': 123}

In [12]:
benchmark_inference("NAFNet")


Loading NAFNet...
[OK] NAFNet: checkpoint loaded successfully.
Checkpoint: /home/sudad/Deburring/Deblurring-SOTA/Results/123-NAFNet_MotionDeblurring_unified/best_ema_generator.pt
Weights key: generator
Strict loading: SUCCESS
Parameters: 67.886529 M
Epoch: 30
Weights type: ema
Trial 1: 61.8122 ms
Trial 2: 62.3915 ms
Trial 3: 62.5871 ms
NAFNet: 62.2636 ± 0.4029 ms
Saved: complexity_results/inference_time/NAFNet_inference.csv
Summary: complexity_results/inference_time/inference_time_summary.csv


{'model': 'NAFNet',
 'inference_mean_ms': 62.263603858947754,
 'inference_std_ms': 0.40294763055099425,
 'inference_median_ms': 62.33166313171387,
 'measured_passes': 300,
 'input_shape': '1x1x352x352',
 'precision': 'FP32',
 'gpu': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU',
 'pytorch': '2.11.0+cu128',
 'cuda': '12.8',
 'warmup_runs': 20,
 'measured_runs': 100,
 'timing_trials': 3,
 'seed': 123}

In [15]:
benchmark_inference("SwinIR-M")


Loading SwinIR-M...


/home/sudad/mamba_env/lib/python3.10/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[OK] SwinIR-M: checkpoint loaded successfully.
Checkpoint: /home/sudad/Deburring/Deblurring-SOTA/Results/123-SwinIR_M_MotionDeblurring_unified/best_ema_generator.pt
Weights key: generator
Strict loading: SUCCESS
Parameters: 11.497681 M
Epoch: 30
Weights type: ema
Trial 1: 1070.5963 ms
Trial 2: 1097.8333 ms
Trial 3: 1092.1289 ms
SwinIR-M: 1086.8528 ± 14.3646 ms
Saved: complexity_results/inference_time/SwinIR_M_inference.csv
Summary: complexity_results/inference_time/inference_time_summary.csv


{'model': 'SwinIR-M',
 'inference_mean_ms': 1086.8528440348307,
 'inference_std_ms': 14.364552617349704,
 'inference_median_ms': 1073.2765502929688,
 'measured_passes': 300,
 'input_shape': '1x1x352x352',
 'precision': 'FP32',
 'gpu': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU',
 'pytorch': '2.11.0+cu128',
 'cuda': '12.8',
 'warmup_runs': 20,
 'measured_runs': 100,
 'timing_trials': 3,
 'seed': 123}

In [10]:
benchmark_inference("Proposed")


Loading Proposed...
[OK] Proposed: checkpoint loaded successfully.
Checkpoint: /home/sudad/Deburring/Deblurring-SOTA/Results/123-Proposed/best_ema_generator.pt
Weights key: gen
Strict loading: SUCCESS
Parameters: 15.425869 M
Epoch: 30
Weights type: ema
Trial 1: 1357.7748 ms
Trial 2: 1370.2060 ms
Trial 3: 1375.3093 ms
Proposed: 1367.7634 ± 9.0188 ms
Saved: complexity_results/inference_time/Proposed_inference.csv
Summary: complexity_results/inference_time/inference_time_summary.csv


{'model': 'Proposed',
 'inference_mean_ms': 1367.763359375,
 'inference_std_ms': 9.018797496549219,
 'inference_median_ms': 1369.9290771484375,
 'measured_passes': 300,
 'input_shape': '1x1x352x352',
 'precision': 'FP32',
 'gpu': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU',
 'pytorch': '2.11.0+cu128',
 'cuda': '12.8',
 'warmup_runs': 20,
 'measured_runs': 100,
 'timing_trials': 3,
 'seed': 123}

In [11]:
summary_path = TIMING_DIR / "inference_time_summary.csv"
if summary_path.exists():
    inference_df = pd.read_csv(summary_path)
    print(inference_df.to_string(index=False))
else:
    print("No inference results have been saved yet.")

    model  inference_mean_ms  inference_std_ms  inference_median_ms  measured_passes input_shape precision                                   gpu      pytorch  cuda  warmup_runs  measured_runs  timing_trials  seed
FFTformer         438.081624          4.179302           432.811050              300 1x1x352x352      FP32 NVIDIA GeForce RTX 5070 Ti Laptop GPU 2.11.0+cu128  12.8           20            100              3   123
     MaIR         970.447988          3.739806           968.553497              300 1x1x352x352      FP32 NVIDIA GeForce RTX 5070 Ti Laptop GPU 2.11.0+cu128  12.8           20            100              3   123
   NAFNet          62.263604          0.402948            62.331663              300 1x1x352x352      FP32 NVIDIA GeForce RTX 5070 Ti Laptop GPU 2.11.0+cu128  12.8           20            100              3   123
 SwinIR-M        1086.852844         14.364553          1073.276550              300 1x1x352x352      FP32 NVIDIA GeForce RTX 5070 Ti Laptop GPU 2.1

In [3]:
#-----------------complexity analysis for ablation study
from pathlib import Path

ABLATION_EXPERIMENTS = {
    "E1": {
        "module_path": "/home/sudad/Deburring/Ablation-123/1/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/abation-1/experiment/best_ema_generator.pt",
    },
    "E2": {
        "module_path": "/home/sudad/Deburring/Ablation-123/2/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-2/experiment/best_ema_generator.pt",
    },
    "E3": {
        "module_path": "/home/sudad/Deburring/Ablation-123/3/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-3/experiment/best_ema_generator.pt",
    },
    "E4": {
        "module_path": "/home/sudad/Deburring/Ablation-123/4/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-4/experiment/best_ema_generator.pt",
    },
    "E5": {
        "module_path": "/home/sudad/Deburring/Ablation-123/5/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-5/experiment/best_ema_generator.pt",
    },
    "E6": {
        "module_path": "/home/sudad/Deburring/Ablation-123/6/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-6/experiment/best_ema_generator.pt",
    },
    "E7": {
        "module_path": "/home/sudad/Deburring/Ablation-123/7/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-7/experiment/best_ema_generator.pt",
    },
    "E8": {
        "module_path": "/home/sudad/Deburring/Ablation-123/8/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-8/experiment/best_ema_generator.pt",
    },
    "E9": {
        "module_path": "/home/sudad/Deburring/Ablation-123/9/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-9/experiment/best_ema_generator.pt",
    },
    "E10": {
        "module_path": "/home/sudad/Deburring/Ablation-123/10/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-10-exper23-1/experiment/best_ema_generator.pt",
    },
    "E11": {
        "module_path": "/home/sudad/Deburring/Ablation-123/11/Module.py",
        "checkpoint_path": "/home/sudad/Deblurring/ablation-11-1/experiment/best_ema_generator.pt",
    },
}

for name, paths in ABLATION_EXPERIMENTS.items():
    module_exists = Path(paths["module_path"]).is_file()
    checkpoint_exists = Path(paths["checkpoint_path"]).is_file()

    print(
        f"{name:4s} | "
        f"Module: {'OK' if module_exists else 'MISSING':7s} | "
        f"Checkpoint: {'OK' if checkpoint_exists else 'MISSING'}"
    )

E1   | Module: OK      | Checkpoint: OK
E2   | Module: OK      | Checkpoint: OK
E3   | Module: OK      | Checkpoint: OK
E4   | Module: OK      | Checkpoint: OK
E5   | Module: OK      | Checkpoint: OK
E6   | Module: OK      | Checkpoint: OK
E7   | Module: OK      | Checkpoint: OK
E8   | Module: OK      | Checkpoint: OK
E9   | Module: OK      | Checkpoint: OK
E10  | Module: OK      | Checkpoint: OK
E11  | Module: OK      | Checkpoint: OK


In [4]:
import ast
import inspect
import torch
from pathlib import Path

GENERATOR_NAMES = [
    "MedMultiwaveletdeblurringGenerator",
    "MedMultiwaveletDeblurringGenerator",
]

for experiment, paths in ABLATION_EXPERIMENTS.items():

    print("\n" + "=" * 80)
    print(experiment)
    print("=" * 80)

    module_path = Path(paths["module_path"])
    checkpoint_path = Path(paths["checkpoint_path"])

    with open(module_path, "r", encoding="utf-8") as f:
        source = f.read()

    tree = ast.parse(source)

    classes = [
        node
        for node in tree.body
        if isinstance(node, ast.ClassDef)
    ]

    generator_classes = [
        node
        for node in classes
        if "generator" in node.name.lower()
    ]

    print("\nGenerator classes:")

    for cls in generator_classes:
        print(" ", cls.name)

        init_method = next(
            (
                node
                for node in cls.body
                if isinstance(
                    node,
                    (ast.FunctionDef, ast.AsyncFunctionDef)
                )
                and node.name == "__init__"
            ),
            None
        )

        if init_method is not None:
            args = init_method.args

            positional = [
                arg.arg
                for arg in args.posonlyargs + args.args
            ]

            keyword_only = [
                arg.arg
                for arg in args.kwonlyargs
            ]

            print("  Constructor arguments:", positional)
            print("  Keyword-only arguments:", keyword_only)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    if not isinstance(checkpoint, dict):
        print("\nCheckpoint type:", type(checkpoint).__name__)
        continue

    print("\nCheckpoint keys:")
    print(list(checkpoint.keys()))

    for key in ["cfg", "config", "train_config"]:

        if key not in checkpoint:
            continue

        config = checkpoint[key]

        if isinstance(config, dict):
            print(f"\nSaved {key}:")

            for config_key, value in config.items():

                if isinstance(
                    value,
                    (str, int, float, bool, type(None))
                ):
                    print(f"  {config_key}: {value}")

        else:
            print(
                f"\nSaved {key} type:",
                type(config).__name__
            )

    del checkpoint

print("\nInspection completed.")


E1

Generator classes:
  MedMultiwaveletDeblurringGenerator
  Constructor arguments: ['self', 'in_ch', 'out_ch', 'wavelet_type', 'band_feature_channels', 'feature_channels', 'cnn_blocks', 'mamba_blocks', 'branch_mode', 'fusion_type', 'use_alignment', 'use_band_refinement', 'use_full_res_refinement']
  Keyword-only arguments: []

Checkpoint keys:
['gen', 'cfg', 'epoch', 'best_psnr', 'weights_type']

Saved cfg:
  train_dir: /home/sudad/relaibility_project/image-70-20-10-2slices/train
  val_dir: /home/sudad/relaibility_project/image-70-20-10-2slices/val
  experiment_name: experiment
  wavelet_type: haar
  branch_mode: cnn
  fusion_type: none
  use_alignment: False
  use_band_refinement: False
  use_full_res_refinement: False
  save_root: /home/sudad/Deblurring/abation-2
  patch_size: 88
  train_motion_length_min: 3
  train_motion_length_max: 15
  motion_angle_min: 0.0
  motion_angle_max: 180.0
  validation_motion_length: 15
  batch_size: 2
  num_workers: 0
  pin_memory: True
  drop_last:

In [5]:
import ast
from pathlib import Path

GENERATOR_NAME = "MedMultiwaveletDeblurringGenerator"

for exp_name, paths in ABLATION_EXPERIMENTS.items():

    module_path = Path(paths["module_path"])

    source = module_path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    generator = next(
        (
            node
            for node in ast.walk(tree)
            if isinstance(node, ast.ClassDef)
            and node.name == GENERATOR_NAME
        ),
        None
    )

    if generator is None:
        print(f"{exp_name}: Generator NOT FOUND")
        continue

    init = next(
        (
            node
            for node in generator.body
            if isinstance(node, ast.FunctionDef)
            and node.name == "__init__"
        ),
        None
    )

    if init is None:
        print(f"{exp_name}: __init__ NOT FOUND")
        continue

    args = init.args.args
    defaults = init.args.defaults

    default_map = {}

    for arg, default in zip(args[-len(defaults):], defaults):
        try:
            default_map[arg.arg] = ast.literal_eval(default)
        except Exception:
            default_map[arg.arg] = ast.unparse(default)


    print(exp_name)
    

    for key in [
        "in_ch",
        "out_ch",
        "wavelet_type",
        "band_feature_channels",
        "feature_channels",
        "cnn_blocks",
        "mamba_blocks",
        "branch_mode",
        "fusion_type",
        "use_alignment",
        "use_band_refinement",
        "use_full_res_refinement",
    ]:
        print(f"{key:28s}: {default_map.get(key, 'REQUIRED')}")


E1
in_ch                       : 1
out_ch                      : 1
wavelet_type                : ghm
band_feature_channels       : 40
feature_channels            : 192
cnn_blocks                  : 8
mamba_blocks                : 8
branch_mode                 : dual
fusion_type                 : gated
use_alignment               : True
use_band_refinement         : True
use_full_res_refinement     : True

E2
in_ch                       : 1
out_ch                      : 1
wavelet_type                : ghm
band_feature_channels       : 40
feature_channels            : 192
cnn_blocks                  : 8
mamba_blocks                : 8
branch_mode                 : dual
fusion_type                 : gated
use_alignment               : True
use_band_refinement         : True
use_full_res_refinement     : True

E3
in_ch                       : 1
out_ch                      : 1
wavelet_type                : ghm
band_feature_channels       : 40
feature_channels            : 192
cnn_blocks   

In [9]:
import os
import sys
import gc
import torch
import importlib.util
from pathlib import Path

GENERATOR_CLASS = "MedMultiwaveletDeblurringGenerator"

ARCH_DEFAULTS = {
    "in_ch": 1,
    "out_ch": 1,
    "band_feature_channels": 40,
    "feature_channels": 192,
    "cnn_blocks": 8,
    "mamba_blocks": 8,
}

ARCH_CFG_KEYS = [
    "wavelet_type",
    "branch_mode",
    "fusion_type",
    "use_alignment",
    "use_band_refinement",
    "use_full_res_refinement",
]


def load_ablation_model(experiment_name, device="cpu"):

    paths = ABLATION_EXPERIMENTS[experiment_name]

    module_path = Path(paths["module_path"])
    checkpoint_path = Path(paths["checkpoint_path"])

    # Import each Module.py under a unique name
    module_name = f"ablation_module_{experiment_name}"

    spec = importlib.util.spec_from_file_location(
        module_name,
        str(module_path)
    )

    if spec is None or spec.loader is None:
        raise ImportError(
            f"Cannot import Module.py for {experiment_name}"
        )

    module = importlib.util.module_from_spec(spec)

    # Allow local imports from the experiment directory
    module_dir = str(module_path.parent)

    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)

    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    Generator = getattr(module, GENERATOR_CLASS)

    # Load checkpoint
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    cfg = checkpoint["cfg"]

    # Architecture arguments
    model_kwargs = ARCH_DEFAULTS.copy()

    for key in ARCH_CFG_KEYS:
        if key not in cfg:
            raise KeyError(
                f"{experiment_name}: Missing architecture key: {key}"
            )

        model_kwargs[key] = cfg[key]

    # Construct generator
    model = Generator(**model_kwargs)

    # Strict checkpoint loading
    model.load_state_dict(
        checkpoint["gen"],
        strict=True
    )

    model = model.to(device)
    model.eval()

    return model, checkpoint, model_kwargs

In [7]:
import pandas as pd

parameter_results = []

for exp_name in ABLATION_EXPERIMENTS:

    print(f"\n{'=' * 65}")
    print(f"Checking {exp_name}")
    print("=" * 65)

    model, checkpoint, model_kwargs = load_ablation_model(
        exp_name,
        device="cpu"
    )

    total_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    params_m = total_params / 1e6

    parameter_results.append({
        "Experiment": exp_name,
        "Parameters": total_params,
        "Parameters_M": params_m,
    })

    print("Architecture:", model_kwargs)
    print(f"Parameters: {total_params:,}")
    print(f"Parameters (M): {params_m:.6f}")
    print("Checkpoint loading: SUCCESS")

    del model, checkpoint
    gc.collect()

parameters_df = pd.DataFrame(parameter_results)

display(parameters_df.style.hide(axis="index"))
parameters_df.to_csv(
    "ablation_parameters.csv",
    index=False
)

print("\nSaved: ablation_parameters.csv")


Checking E1


/home/sudad/mamba_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Architecture: {'in_ch': 1, 'out_ch': 1, 'band_feature_channels': 40, 'feature_channels': 192, 'cnn_blocks': 8, 'mamba_blocks': 8, 'wavelet_type': 'haar', 'branch_mode': 'cnn', 'fusion_type': 'none', 'use_alignment': False, 'use_band_refinement': False, 'use_full_res_refinement': False}
Parameters: 6,637,724
Parameters (M): 6.637724
Checkpoint loading: SUCCESS

Checking E2
Architecture: {'in_ch': 1, 'out_ch': 1, 'band_feature_channels': 40, 'feature_channels': 192, 'cnn_blocks': 8, 'mamba_blocks': 8, 'wavelet_type': 'haar', 'branch_mode': 'mamba', 'fusion_type': 'none', 'use_alignment': False, 'use_band_refinement': False, 'use_full_res_refinement': False}
Parameters: 3,341,468
Parameters (M): 3.341468
Checkpoint loading: SUCCESS

Checking E3
Architecture: {'in_ch': 1, 'out_ch': 1, 'band_feature_channels': 40, 'feature_channels': 192, 'cnn_blocks': 8, 'mamba_blocks': 8, 'wavelet_type': 'haar', 'branch_mode': 'dual', 'fusion_type': 'average', 'use_alignment': False, 'use_band_refinement'

Experiment,Parameters,Parameters_M
E1,6637724,6.637724
E2,3341468,3.341468
E3,8984924,8.984924
E4,10644572,10.644572
E5,12563336,12.563336
E6,13166025,13.166025
E7,12683914,12.683914
E8,14702602,14.702602
E9,13286603,13.286603
E10,15425869,15.425869



Saved: ablation_parameters.csv


In [8]:
# ============================================================
# ABLATION STUDY — INFERENCE TIME MEASUREMENT
# Same protocol as SOTA computational efficiency evaluation
# ============================================================

import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Timing configuration
# ============================================================

DEVICE = torch.device("cuda")

INPUT_SHAPE = (1, 1, 352, 352)

WARMUP_ITERATIONS = 20
MEASUREMENT_ITERATIONS = 100
NUM_TRIALS = 3

OUTPUT_DIR = Path("ablation_complexity_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Verify CUDA environment
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for inference timing.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

# Match the previous FP32 measurement protocol
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# ============================================================
# 3. Inference time measurement function
# ============================================================

def measure_ablation_inference_time(experiment_name):

    print("\n" + "=" * 75)
    print(f"INFERENCE TIME — {experiment_name}")
    print("=" * 75)

    # --------------------------------------------------------
    # Load the exact architecture and trained checkpoint
    # --------------------------------------------------------

    model, checkpoint, model_kwargs = load_ablation_model(
        experiment_name,
        device=DEVICE
    )

    model.eval()

    # --------------------------------------------------------
    # Count trainable parameters
    # --------------------------------------------------------

    total_parameters = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    parameters_m = total_parameters / 1e6

    # --------------------------------------------------------
    # Prepare FP32 input
    # --------------------------------------------------------

    x = torch.randn(
        INPUT_SHAPE,
        device=DEVICE,
        dtype=torch.float32
    )

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    with torch.inference_mode():

        for _ in range(WARMUP_ITERATIONS):
            _ = model(x)

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Timing trials
        # ----------------------------------------------------

        trial_times = []

        for trial in range(NUM_TRIALS):

            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)

            start_event.record()

            for _ in range(MEASUREMENT_ITERATIONS):
                _ = model(x)

            end_event.record()

            torch.cuda.synchronize()

            elapsed_ms = start_event.elapsed_time(end_event)

            average_ms = elapsed_ms / MEASUREMENT_ITERATIONS

            trial_times.append(average_ms)

            print(
                f"Trial {trial + 1}: "
                f"{average_ms:.6f} ms/image"
            )

    # --------------------------------------------------------
    # Mean and sample standard deviation
    # --------------------------------------------------------

    mean_ms = float(np.mean(trial_times))

    std_ms = float(
        np.std(trial_times, ddof=1)
    )

    print("\nRESULTS")
    print("-" * 75)

    print(f"Experiment: {experiment_name}")
    print(f"Parameters: {total_parameters:,}")
    print(f"Parameters (M): {parameters_m:.6f}")
    print(f"Inference Time: {mean_ms:.6f} ± {std_ms:.6f} ms")

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    result = {
        "Experiment": experiment_name,
        "Parameters": total_parameters,
        "Parameters_M": parameters_m,
        "Inference_Time_Mean_ms": mean_ms,
        "Inference_Time_STD_ms": std_ms,
        "Trial_1_ms": trial_times[0],
        "Trial_2_ms": trial_times[1],
        "Trial_3_ms": trial_times[2],
    }

    result_path = OUTPUT_DIR / f"{experiment_name}_inference_time.csv"

    pd.DataFrame([result]).to_csv(
        result_path,
        index=False
    )

    print("\nSaved:", result_path)

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del model
    del checkpoint
    del x

    gc.collect()

    torch.cuda.empty_cache()

    return result


print("\nAblation timing function is ready.")

GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
PyTorch: 2.11.0+cu128
CUDA: 12.8

Ablation timing function is ready.


In [10]:
result_E1 = measure_ablation_inference_time("E1")


INFERENCE TIME — E1
Trial 1: 276.270312 ms/image
Trial 2: 276.837676 ms/image
Trial 3: 276.748789 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E1
Parameters: 6,637,724
Parameters (M): 6.637724
Inference Time: 276.618926 ± 0.305162 ms

Saved: ablation_complexity_results/E1_inference_time.csv


In [11]:
result_E2 = measure_ablation_inference_time("E2")


INFERENCE TIME — E2
Trial 1: 169.986602 ms/image
Trial 2: 170.220723 ms/image
Trial 3: 170.418125 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E2
Parameters: 3,341,468
Parameters (M): 3.341468
Inference Time: 170.208483 ± 0.216022 ms

Saved: ablation_complexity_results/E2_inference_time.csv


In [12]:
result_E3 = measure_ablation_inference_time("E3")


INFERENCE TIME — E3
Trial 1: 424.721055 ms/image
Trial 2: 427.710820 ms/image
Trial 3: 427.096797 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E3
Parameters: 8,984,924
Parameters (M): 8.984924
Inference Time: 426.509557 ± 1.579023 ms

Saved: ablation_complexity_results/E3_inference_time.csv


In [13]:
result_E4 = measure_ablation_inference_time("E4")


INFERENCE TIME — E4
Trial 1: 501.767656 ms/image
Trial 2: 503.879531 ms/image
Trial 3: 503.801523 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E4
Parameters: 10,644,572
Parameters (M): 10.644572
Inference Time: 503.149570 ± 1.197408 ms

Saved: ablation_complexity_results/E4_inference_time.csv


In [14]:
result_E5 = measure_ablation_inference_time("E5")


INFERENCE TIME — E5
Trial 1: 1060.092109 ms/image
Trial 2: 1061.735938 ms/image
Trial 3: 1063.326250 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E5
Parameters: 12,563,336
Parameters (M): 12.563336
Inference Time: 1061.718099 ± 1.617144 ms

Saved: ablation_complexity_results/E5_inference_time.csv


In [15]:
result_E6 = measure_ablation_inference_time("E6")


INFERENCE TIME — E6
Trial 1: 1199.240859 ms/image
Trial 2: 1199.028828 ms/image
Trial 3: 1197.237969 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E6
Parameters: 13,166,025
Parameters (M): 13.166025
Inference Time: 1198.502552 ± 1.100281 ms

Saved: ablation_complexity_results/E6_inference_time.csv


In [16]:
result_E7 = measure_ablation_inference_time("E7")


INFERENCE TIME — E7
Trial 1: 1081.360312 ms/image
Trial 2: 1079.110312 ms/image
Trial 3: 1081.042578 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E7
Parameters: 12,683,914
Parameters (M): 12.683914
Inference Time: 1080.504401 ± 1.217724 ms

Saved: ablation_complexity_results/E7_inference_time.csv


In [17]:
result_E8 = measure_ablation_inference_time("E8")


INFERENCE TIME — E8
Trial 1: 1228.425234 ms/image
Trial 2: 1233.025469 ms/image
Trial 3: 1232.415859 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E8
Parameters: 14,702,602
Parameters (M): 14.702602
Inference Time: 1231.288854 ± 2.498629 ms

Saved: ablation_complexity_results/E8_inference_time.csv


In [18]:
result_E9 = measure_ablation_inference_time("E9")


INFERENCE TIME — E9
Trial 1: 1212.927344 ms/image
Trial 2: 1210.427969 ms/image
Trial 3: 1211.268750 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E9
Parameters: 13,286,603
Parameters (M): 13.286603
Inference Time: 1211.541354 ± 1.271792 ms

Saved: ablation_complexity_results/E9_inference_time.csv


In [19]:
result_E10 = measure_ablation_inference_time("E10")


INFERENCE TIME — E10
Trial 1: 1378.042500 ms/image
Trial 2: 1374.392031 ms/image
Trial 3: 1377.771719 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E10
Parameters: 15,425,869
Parameters (M): 15.425869
Inference Time: 1376.735417 ± 2.033942 ms

Saved: ablation_complexity_results/E10_inference_time.csv


In [20]:
result_E11 = measure_ablation_inference_time("E11")


INFERENCE TIME — E11
Trial 1: 1376.895937 ms/image
Trial 2: 1374.277969 ms/image
Trial 3: 1373.966719 ms/image

RESULTS
---------------------------------------------------------------------------
Experiment: E11
Parameters: 15,425,869
Parameters (M): 15.425869
Inference Time: 1375.046875 ± 1.608879 ms

Saved: ablation_complexity_results/E11_inference_time.csv


In [21]:
# ============================================================
# COMBINE ABLATION RESULTS — E1 TO E11
# ============================================================

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("ablation_complexity_results")

all_results = []

for i in range(1, 12):

    experiment_name = f"E{i}"

    result_path = (
        OUTPUT_DIR / f"{experiment_name}_inference_time.csv"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            f"Missing inference results for {experiment_name}: "
            f"{result_path}"
        )

    df = pd.read_csv(result_path)

    all_results.append(df)

# Combine in E1 → E11 order
ablation_results_df = pd.concat(
    all_results,
    ignore_index=True
)

# Select columns
ablation_results_df = ablation_results_df[
    [
        "Experiment",
        "Parameters",
        "Parameters_M",
        "Inference_Time_Mean_ms",
        "Inference_Time_STD_ms",
        "Trial_1_ms",
        "Trial_2_ms",
        "Trial_3_ms",
    ]
]

# Save full results
output_path = OUTPUT_DIR / "ablation_complexity_summary.csv"

ablation_results_df.to_csv(
    output_path,
    index=False
)

print("\nABLATION COMPLEXITY SUMMARY")
print("=" * 90)

display(
    ablation_results_df.style.hide(axis="index").format(
        {
            "Parameters_M": "{:.6f}",
            "Inference_Time_Mean_ms": "{:.6f}",
            "Inference_Time_STD_ms": "{:.6f}",
            "Trial_1_ms": "{:.6f}",
            "Trial_2_ms": "{:.6f}",
            "Trial_3_ms": "{:.6f}",
        }
    )
)

print("\nSaved:", output_path)


ABLATION COMPLEXITY SUMMARY


Experiment,Parameters,Parameters_M,Inference_Time_Mean_ms,Inference_Time_STD_ms,Trial_1_ms,Trial_2_ms,Trial_3_ms
E1,6637724,6.637724,276.618926,0.305162,276.270312,276.837676,276.748789
E2,3341468,3.341468,170.208483,0.216022,169.986602,170.220723,170.418125
E3,8984924,8.984924,426.509557,1.579023,424.721055,427.710820,427.096797
E4,10644572,10.644572,503.149570,1.197408,501.767656,503.879531,503.801523
E5,12563336,12.563336,1061.718099,1.617144,1060.092109,1061.735938,1063.326250
E6,13166025,13.166025,1198.502552,1.100281,1199.240859,1199.028828,1197.237969
E7,12683914,12.683914,1080.504401,1.217724,1081.360312,1079.110312,1081.042578
E8,14702602,14.702602,1231.288854,2.498629,1228.425234,1233.025469,1232.415859
E9,13286603,13.286603,1211.541354,1.271792,1212.927344,1210.427969,1211.268750
E10,15425869,15.425869,1376.735417,2.033942,1378.042500,1374.392031,1377.771719



Saved: ablation_complexity_results/ablation_complexity_summary.csv


In [22]:
# ============================================================
# PUBLICATION-READY ABLATION TABLE
# ============================================================

publication_df = ablation_results_df.copy()

publication_df["Parameters (M)"] = (
    publication_df["Parameters_M"]
    .map(lambda x: f"{x:.3f}")
)

publication_df["Inference Time (ms)"] = (
    publication_df.apply(
        lambda row:
        f"{row['Inference_Time_Mean_ms']:.3f} ± "
        f"{row['Inference_Time_STD_ms']:.3f}",
        axis=1
    )
)

publication_df = publication_df[
    [
        "Experiment",
        "Parameters (M)",
        "Inference Time (ms)",
    ]
]

display(
    publication_df.style.hide(axis="index")
)

publication_df.to_csv(
    OUTPUT_DIR / "ablation_publication_table.csv",
    index=False
)

print("Publication table saved.")

Experiment,Parameters (M),Inference Time (ms)
E1,6.638,276.619 ± 0.305
E2,3.341,170.208 ± 0.216
E3,8.985,426.510 ± 1.579
E4,10.645,503.150 ± 1.197
E5,12.563,1061.718 ± 1.617
E6,13.166,1198.503 ± 1.100
E7,12.684,1080.504 ± 1.218
E8,14.703,1231.289 ± 2.499
E9,13.287,1211.541 ± 1.272
E10,15.426,1376.735 ± 2.034


Publication table saved.
